# Project 1: E-commerce Delivery & Logistics Performance Analysis
## Week 3: Master Operational Opportunity Assessment & Prioritized Issue Engine
**Lead Analyst**: Aman Shakil  
**Deliverable**: Comprehensive Week 3 Notebook (Root-Cause Prioritization, Policy Simulations, and Actionable ROI)

This notebook synthesizes Week 3's engineering tasks:
1. Quantifying recurring bottlenecks across order volume, delay severity, and customer dissatisfaction.
2. Building the **Business-Weighted Prioritization Algorithm**.
3. Simulating four high-ROI operational interventions (Merchant 48h SLA, Rio de Janeiro Carrier Buffer, Friday 2 PM Packaging Cutoff, and Bulky Freight Dynamic Delivery Promise Buffers).
4. Exporting the actionable prioritization dataset and ROI models for the Week 4 Power BI dashboard.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)

# Load validated KPI dataset
kpi_df = pd.read_csv('../data/processed/validated_kpi_dataset.csv')
valid = kpi_df[kpi_df['is_eligible_otd'] == 1].copy()
print(f"Loaded {len(kpi_df):,} placed checkout orders ({len(valid):,} eligible delivered orders).")


## 1. Prioritization Engine: Scoring Methodology
We formalize a composite Priority Score that balances volume, delay severity, and customer dissatisfaction:
$$\text{Priority Score} = \frac{\text{Delayed Orders} \times \text{Avg Delay Days} \times (1 + \max(0, \text{Baseline Rating} - \text{Segment Rating}))}{100}$$


In [ ]:
# Run prioritization scoring engine
from sys import path
path.append('../src')
from prioritize_issues import compute_prioritized_issues

issue_df = compute_prioritized_issues('..')
display(issue_df[['Issue ID', 'Issue Name', 'Volume Affected', 'Delayed Orders', 'Avg Delay (Days)', 'Priority Score']])


## 2. Root-Cause Analysis Across the 5 Major Bottlenecks

### 2.1 Issue 1: Chronic Merchant Packaging Lags (>4.0 Days)


In [ ]:
chronic = valid[valid['order_processing_time_days'] > 4.0]
fast = valid[valid['order_processing_time_days'] <= 2.0]

print(f"Chronic Sellers Volume: {len(chronic):,} orders ({len(chronic)/len(valid)*100:.1f}%)")
print(f"Chronic Sellers OTD:    {(chronic['delivery_outcome'] == 'on_time').mean()*100:.2f}% (vs. Fast Sellers: {(fast['delivery_outcome'] == 'on_time').mean()*100:.2f}%)")
print(f"Revenue at Risk:        R$ {chronic[chronic['delivery_outcome'] == 'delayed']['total_order_value'].sum():,.2f}")


### 2.2 Issue 2: Rio de Janeiro (RJ) Metropolitan Carrier Congestion


In [ ]:
rj_orders = valid[valid['customer_state'] == 'RJ']
sp_orders = valid[valid['customer_state'] == 'SP']

print(f"RJ Delayed Orders:  {(rj_orders['delivery_outcome'] == 'delayed').sum():,} (Delay Rate: {(rj_orders['delivery_outcome'] == 'delayed').mean()*100:.2f}%)")
print(f"SP Delayed Orders:  {(sp_orders['delivery_outcome'] == 'delayed').sum():,} (Delay Rate: {(sp_orders['delivery_outcome'] == 'delayed').mean()*100:.2f}%)")
print(f"RJ Customer Rating: {rj_orders['review_score'].mean():.2f} ★ (vs. SP: {sp_orders['review_score'].mean():.2f} ★)")


### 2.3 Issue 3: Friday & Saturday Weekend Packaging Freeze


In [ ]:
weekend_orders = valid[valid['purchase_weekday'].isin(['Friday', 'Saturday'])]
tuesday_orders = valid[valid['purchase_weekday'] == 'Tuesday']

print(f"Friday/Saturday Orders: {len(weekend_orders):,} (Mean Processing: {weekend_orders['order_processing_time_days'].mean():.2f} days)")
print(f"Tuesday Orders:         {len(tuesday_orders):,} (Mean Processing: {tuesday_orders['order_processing_time_days'].mean():.2f} days)")
print(f"Weekend Latency Added:  +{weekend_orders['order_processing_time_days'].mean() - tuesday_orders['order_processing_time_days'].mean():.2f} days")


## 3. Operational Policy Simulations: Baseline vs. Projected OTD Uplift

We simulate the quantitative impact of the 4 core operational recommendations:
* **REC-01**: 48-Hour Merchant SLA (capping processing at 2.0 days).
* **REC-02**: RJ Regional Drop-Shipping & Carrier Diversification (reducing RJ transit by 3.5 days).
* **REC-03**: Friday 2:00 PM Packaging Cut-off & Saturday Morning Sweeps (-1.1 days latency).
* **REC-04**: Dynamic Promised Delivery Buffering for Bulky Freight (+4 business days).


In [ ]:
# Run visualization and simulation script
from generate_week3_visualizations import generate_week3_charts
generate_week3_charts('..')
print("Simulations complete. Visual outputs saved in reports/figures/.")


## 4. Summary of Operational Opportunities for Week 4 Dashboard
These actionable findings and prioritized issue ranks form the core diagnostic drill-downs for the interactive Power BI dashboard in Week 4:
1. **Executive Overview**: Total Orders, OTD Rate, and Top Bottleneck Severity Cards.
2. **Merchant SLA Diagnostics**: Seller dispatch latency distribution and compliance tier filters.
3. **Regional Route Corridor Heatmap**: RJ/BA last-mile bottlenecks with simulated policy toggles.
4. **Customer Rating Decay Visualizer**: Real-time correlation between delay severity and 1-star review risk.
